# Appendix M — Enhanced Recognizer and Lexicon-Guided Neuro-Symbolic Validation (Execution Notebook)

Second, pre-specified study following Appendix K. Same RxHandBD partition (training 3,125 / validation 669 /
holdout 669) and same primary scorer. **Every choice is made on the validation partition and frozen (hashed)
before the holdout or the original test partition is processed.** The holdout was opened once before, for
Appendix K; this is its second use and is reported as such.

| Stage | Pre-specified setting |
|---|---|
| Recognizer trials | base-handwritten @1e-5; large-handwritten @1e-5; large-handwritten @5e-6 |
| Images | aspect-preserving crop-and-pad (Appendix K) + training-only augmentation |
| Augmentation (p = 0.8) | affine ±3°, shear ±5°, scale 0.90–1.05, translate 3%; brightness/contrast ±30%; blur p 0.2; stroke thin/thicken p 0.2 |
| Optimization | AdamW, wd 0.01, 5% warmup + linear decay, batch 8 × accumulation 2, bf16, clip 1.0, seed 42 |
| Epochs | up to 20, early stopping on validation exact (patience 4) |
| Recognizer selection | validation exact (WER tie-break); then greedy vs. beam 4 on validation |
| LGC candidates | 10-best beam list ∪ 20 nearest lexicon entries (normalized edit distance) |
| LGC scoring | log P(candidate \| image) [sum or per-token] + α·in-lexicon + β·log(1+frequency) |
| LGC grid (validation) | α ∈ {0,1,2,4,8}, β ∈ {0,0.5,1}, mode ∈ {sum, mean} |
| C3 acceptance | LGC output in lexicon, no implemented review flag, score margin ≥ θ; θ = smallest value whose validation accepted precision ≥ that of C1 |
| Lexicon | validation tuning: training labels only; holdout and original test: training + validation labels |

In [ ]:
!pip -q install "transformers==4.57.1" "accelerate>=0.34" safetensors rapidfuzz
import torch, transformers, sys
print('torch', torch.__version__, '| transformers', transformers.__version__, '| python', sys.version.split()[0])
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'
print('GPU:', torch.cuda.get_device_name(0), '| bf16', torch.cuda.is_bf16_supported())

In [ ]:
# ---- Configuration (fixed before the first trial) ----
from google.colab import drive
drive.mount('/content/drive')
WORK_K = '/content/drive/MyDrive/praxis_appendix_k'     # archive, manifests and Appendix K holdout predictions
WORK = '/content/drive/MyDrive/praxis_appendix_m'       # everything produced by this study
import os; os.makedirs(WORK, exist_ok=True)
TRIALS = [  # (name, model id, learning rate)
    ('base_lr1e-05', 'microsoft/trocr-base-handwritten', 1e-5),
    ('large_lr1e-05', 'microsoft/trocr-large-handwritten', 1e-5),
    ('large_lr5e-06', 'microsoft/trocr-large-handwritten', 5e-6),
]
BATCH, ACCUM, MAX_EPOCHS, WARMUP, WD, PATIENCE, SEED = 8, 2, 20, 0.05, 0.01, 4, 42
EVAL_BATCH, MAX_NEW_TOKENS, MAX_LABEL_TOKENS = 32, 64, 64
NBEST, KLEX = 10, 20
TARGET = 0.80
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

In [ ]:
# ---- Appendix K utilities (unchanged): normalization, scoring, image handling, manifests ----
import csv, hashlib, json, math, os, re, unicodedata, zipfile
from pathlib import Path
import numpy as np
from PIL import Image


def normalize(s):
    """Primary scorer (Section 4.3): NFC, collapse whitespace, case-fold; punctuation and numerals kept."""
    s = unicodedata.normalize('NFC', s or '')
    s = re.sub(r'\s+', ' ', s).strip()
    return s.casefold()


def edit_counts(ref, hyp):
    """Deterministic Levenshtein alignment -> substitutions, deletions, insertions, hits, N."""
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        d[i][0] = i
    for j in range(1, m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            c = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i][j] = min(d[i - 1][j - 1] + c, d[i - 1][j] + 1, d[i][j - 1] + 1)
    S = D = I = H = 0
    i, j = n, m
    while i > 0 or j > 0:   # fixed tie-break order: match/sub, deletion, insertion
        if i > 0 and j > 0 and d[i][j] == d[i - 1][j - 1] + (0 if ref[i - 1] == hyp[j - 1] else 1):
            if ref[i - 1] == hyp[j - 1]:
                H += 1
            else:
                S += 1
            i, j = i - 1, j - 1
        elif i > 0 and d[i][j] == d[i - 1][j] + 1:
            D += 1; i -= 1
        else:
            I += 1; j -= 1
    return {'S': S, 'D': D, 'I': I, 'H': H, 'N': n}


def score(refs, hyps):
    """Micro CER/WER (pooled), complete-label exact agreement. Empty outputs stay in the denominator."""
    assert len(refs) == len(hyps)
    ch = {'S': 0, 'D': 0, 'I': 0, 'H': 0, 'N': 0}
    wd = {'S': 0, 'D': 0, 'I': 0, 'H': 0, 'N': 0}
    exact = 0
    for r, h in zip(refs, hyps):
        r, h = normalize(r), normalize(h)
        exact += int(r == h)
        for k, v in edit_counts(list(r), list(h)).items():
            ch[k] += v
        for k, v in edit_counts(r.split(' ') if r else [], h.split(' ') if h else []).items():
            wd[k] += v
    n = len(refs)
    return {
        'n': n, 'exact_count': exact, 'exact': exact / n if n else None,
        'cer': (ch['S'] + ch['D'] + ch['I']) / ch['N'] if ch['N'] else None,
        'wer': (wd['S'] + wd['D'] + wd['I']) / wd['N'] if wd['N'] else None,
        'char_counts': ch, 'word_counts': wd,
    }


def otsu(gray):
    hist = np.bincount(gray.ravel(), minlength=256).astype(float)
    total = gray.size
    sum_all = np.dot(np.arange(256), hist)
    wb = sb = 0.0
    best, thr = -1.0, 127
    for t in range(256):
        wb += hist[t]
        if wb == 0:
            continue
        wf = total - wb
        if wf == 0:
            break
        sb += t * hist[t]
        mb, mf = sb / wb, (sum_all - sb) / wf
        between = wb * wf * (mb - mf) ** 2
        if between > best:
            best, thr = between, t
    return thr


def crop_and_pad(img, margin=8):
    """Appendix K 'aspect-preserving crop-and-pad': crop to the ink bounding box (Otsu), pad to a white square."""
    rgb = img.convert('RGB')
    g = np.asarray(rgb.convert('L'))
    thr = min(otsu(g), float(np.median(g)) - 40)   # guard against splitting background noise
    ink = g <= thr
    rows = np.where(ink.sum(axis=1) >= 2)[0]
    cols = np.where(ink.sum(axis=0) >= 2)[0]
    if ink.sum() < 10 or len(rows) == 0 or len(cols) == 0:
        return rgb
    x0, x1 = max(0, cols.min() - margin), min(g.shape[1], cols.max() + 1 + margin)
    y0, y1 = max(0, rows.min() - margin), min(g.shape[0], rows.max() + 1 + margin)
    c = rgb.crop((x0, y0, x1, y1))
    side = max(c.size)
    canvas = Image.new('RGB', (side, side), (255, 255, 255))
    canvas.paste(c, ((side - c.size[0]) // 2, (side - c.size[1]) // 2))
    return canvas


def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def is_scorable(ref):
    return not re.fullmatch(r'[?\s]+', ref or '')


def find_split_csv(root, split):
    hits = [p for p in Path(root).rglob('*.csv')
            if re.sub(r'[^a-z]', '', p.stem.lower()) in {split + 'label', split + 'labels'}]
    if len(hits) != 1:
        raise ValueError(f'Expected one {split} label CSV, found {len(hits)}: {hits}')
    return hits[0]


def read_rows(csv_path):
    with open(csv_path, encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    cols = {c.lower().strip(): c for c in rows[0].keys()}
    ic = next(cols[c] for c in ('images', 'image', 'image_name', 'filename', 'file') if c in cols)
    tc = next(cols[c] for c in ('text', 'label', 'transcription') if c in cols)
    return [(r[ic].strip(), r[tc].strip()) for r in rows]


def index_images(root):
    idx = {}
    for p in Path(root).rglob('*'):
        if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}:
            idx.setdefault(p.name, []).append(p)
    return idx


def resolve(rows, idx):
    out = []
    for name, ref in rows:
        c = idx.get(Path(name.replace('\\', '/')).name, [])
        if len(c) != 1:
            raise ValueError(f'Missing or ambiguous image: {name} ({len(c)} matches)')
        out.append({'record_id': c[0].name, 'image_path': str(c[0]), 'reference_text': ref})
    return out

In [ ]:
# ===== Lexicon-guided correction (LGC) and the four validation conditions (pure Python) =====
# Pre-specified before any v2 output was produced. Everything that is tuned is tuned on the validation
# partition with the training-label lexicon only; the frozen values are then applied unchanged.
import math, random, re, unicodedata, json
from collections import Counter

def normalize(s):
    s = unicodedata.normalize('NFC', s or '')
    return re.sub(r'\s+', ' ', s).strip().casefold()

# ---- implemented numeric review rules (Appendix F, unchanged from word_ablation.py) ----
ATOM = (r"[+\-−]?(?:\d+\s+\d+\s*/\s*\d+|\d+\s*/\s*\d+"
        r"|\d*[.,]\d+(?:[eE][+\-]?\d+)?|\d+[eE][+\-]?\d+"
        r"|\d+[¼½¾]|\d+|[¼½¾])")
NUMBER = r"(?:" + ATOM + r"\s*[–−-]\s*" + ATOM + r"|" + ATOM + r")"

def numeric_review_flags(value):
    flags = []
    if re.match(r'[+\-−]', value): flags.append('signed_quantity')
    if re.search(r'\d[eE][+\-]?\d', value): flags.append('scientific_quantity')
    if re.search(r'[\d¼½¾]\s*[–−-]\s*[+\-−]?\d', value): flags.append('quantity_range')
    if re.search(r'\d*,\d', value): flags.append('decimal_comma_quantity')
    if re.search(r'/\s*0(?:\D|$)', value): flags.append('zero_denominator')
    return flags

class Lexicon:
    """Normalized label -> raw spellings and frequency, from development labels only."""
    def __init__(self, labels):
        self.raw, self.freq = {}, Counter()
        for t in labels:
            n = normalize(t)
            if not n or re.fullmatch(r'[?\s]+', n):
                continue
            self.raw.setdefault(n, Counter())[t.strip()] += 1
            self.freq[n] += 1
        self.keys = sorted(self.raw)
    def __contains__(self, n): return n in self.raw
    def spelling(self, n): return self.raw[n].most_common(1)[0][0]
    def nearest(self, text, k):
        """k lexicon entries with the smallest normalized edit distance to text (ties: frequency, then alphabet)."""
        t = normalize(text)
        scored = [(lev(t, e) / max(len(t), len(e), 1), -self.freq[e], e) for e in self.keys]
        scored.sort()
        return [e for _, _, e in scored[:k]]

try:
    from rapidfuzz.distance import Levenshtein as _RL   # same unit-cost distance, faster
except Exception:
    _RL = None

def lev(a, b):
    if _RL is not None: return _RL.distance(a, b)
    if a == b: return 0
    if len(a) < len(b): a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def gate(hyp, lex, full):
    """C1 (full=False) and C2 (full=True) acceptance rules, identical to the executed word-level ablation."""
    h = normalize(hyp); flags = []
    if h not in lex:
        flags.append('terminology_unresolved')
    if full:
        for m in re.finditer(NUMBER, hyp):
            flags += numeric_review_flags(m.group(0))
        tokens = h.split(' ') if h else []
        if len({t for t in tokens if t in lex and len(tokens) > 1}) > 1:
            flags.append('ambiguous_medication_count')
    return len(flags) == 0, sorted(set(flags))

# ---- LGC: rescoring the recognizer's n-best list together with nearby lexicon entries ----
ALPHAS = [0.0, 1.0, 2.0, 4.0, 8.0]          # bonus for a candidate that is a lexicon entry
BETAS = [0.0, 0.5, 1.0]                     # weight on log(1 + lexicon frequency)
LENGTH_MODES = ['sum', 'mean']              # total or per-token log-likelihood
THETAS = [0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]   # minimum score margin for C3 acceptance

def cand_score(c, lex, alpha, beta, mode):
    n = normalize(c['text'])
    lp = c['logp'] if mode == 'sum' else c['logp'] / max(c['ntok'], 1)
    inlex = n in lex
    return lp + alpha * inlex + beta * math.log1p(lex.freq[n] if inlex else 0)

def lgc_choose(rec, lex, alpha, beta, mode):
    """Return (chosen text, margin). Candidates are deduplicated by normalized form, keeping the best score."""
    best = {}
    for c in rec['cands']:
        n = normalize(c['text'])
        if not n:
            continue
        s = cand_score(c, lex, alpha, beta, mode)
        if n not in best or s > best[n][0]:
            best[n] = (s, c['text'])
    if not best:
        return rec['top1'], 0.0
    ranked = sorted(best.values(), key=lambda x: -x[0])
    margin = ranked[0][0] - ranked[1][0] if len(ranked) > 1 else float('inf')
    return ranked[0][1], margin

def exact_rate(recs, outs):
    return sum(normalize(o) == normalize(r['ref']) for r, o in zip(recs, outs)) / len(recs)

def select_lgc(val_recs, lex):
    """Grid in simplicity order; strictly better validation exact agreement is required to move on."""
    grid = [(a, b, m) for m in LENGTH_MODES for a in ALPHAS for b in BETAS]
    table, best = [], None
    for a, b, m in grid:
        outs = [lgc_choose(r, lex, a, b, m)[0] for r in val_recs]
        e = exact_rate(val_recs, outs)
        table.append({'alpha': a, 'beta': b, 'mode': m, 'val_exact': e})
        if best is None or e > best['val_exact']:
            best = table[-1]
    return best, table

def c3_decisions(recs, lex, p, theta):
    out = []
    for r in recs:
        txt, margin = lgc_choose(r, lex, p['alpha'], p['beta'], p['mode'])
        ok_full, flags = gate(txt, lex, full=True)
        out.append((txt, ok_full and margin >= theta, margin, flags))
    return out

def select_theta(val_recs, lex, p):
    """Smallest margin threshold whose validation accepted precision is at least that of the exact lookup (C1)."""
    acc1 = [r for r in val_recs if gate(r['top1'], lex, False)[0]]
    target = sum(normalize(r['top1']) == normalize(r['ref']) for r in acc1) / len(acc1) if acc1 else 1.0
    rows = []
    for th in THETAS:
        d = c3_decisions(val_recs, lex, p, th)
        acc = [(r, t) for r, (t, a, _, _) in zip(val_recs, d) if a]
        prec = sum(normalize(t) == normalize(r['ref']) for r, t in acc) / len(acc) if acc else None
        rows.append({'theta': th, 'accepted': len(acc), 'precision': prec,
                     'recall': sum(normalize(t) == normalize(r['ref']) for r, t in acc) / len(val_recs)})
    ok = [x for x in rows if x['precision'] is not None and x['precision'] >= target]
    chosen = ok[0] if ok else max((x for x in rows if x['precision'] is not None), key=lambda x: (x['precision'], -x['theta']))
    return chosen['theta'], target, rows

# ---- evaluation of the conditions on one population ----
def build_records(recs, lex, p, theta):
    d = c3_decisions(recs, lex, p, theta)
    out = []
    for r, (txt, acc3, margin, flags) in zip(recs, d):
        c1, _ = gate(r['top1'], lex, False)
        c2, f2 = gate(r['top1'], lex, True)
        out.append({'id': r['id'], 'ref': r['ref'], 'top1': r['top1'], 'lgc': txt, 'margin': margin,
                    'top1_correct': normalize(r['top1']) == normalize(r['ref']),
                    'lgc_correct': normalize(txt) == normalize(r['ref']),
                    'ref_in_lexicon': normalize(r['ref']) in lex,
                    'C0': True, 'C1': c1, 'C2': c2, 'C3': acc3, 'C2_flags': f2, 'C3_flags': flags})
    return out

def cond_metrics(recs, cond):
    key = 'lgc_correct' if cond == 'C3' else 'top1_correct'
    n = len(recs); acc = [r for r in recs if r[cond]]; ok = sum(r[key] for r in acc)
    wd = [r for r in recs if not r[cond]]
    return {'n': n, 'accepted': len(acc), 'coverage': len(acc) / n, 'accepted_correct': ok,
            'incorrect_accepted': len(acc) - ok, 'accepted_precision': ok / len(acc) if acc else None,
            'incorrect_accepted_rate_of_all': (len(acc) - ok) / n, 'whole_test_recall': ok / n,
            'review_rate': len(wd) / n}

def wilson(k, n, z=1.959963984540054):
    if not n: return [None, None]
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return [c - h, c + h]

def paired_bootstrap(recs, a, b, key, B=2000, seed=42):
    full_a, full_b = cond_metrics(recs, a)[key], cond_metrics(recs, b)[key]
    if full_a is None or full_b is None:   # e.g. a condition that accepted nothing: precision undefined
        return {'diff': None, 'ci95': [None, None], 'replicates': 0}
    rng = random.Random(seed); n = len(recs); diffs = []
    for _ in range(B):
        s = [recs[rng.randrange(n)] for _ in range(n)]
        ma, mb = cond_metrics(s, a)[key], cond_metrics(s, b)[key]
        if ma is not None and mb is not None:
            diffs.append(ma - mb)
    diffs.sort()
    if not diffs:
        return {'diff': full_a - full_b, 'ci95': [None, None], 'replicates': 0}
    return {'diff': full_a - full_b,
            'ci95': [diffs[int(0.025 * len(diffs))], diffs[int(0.975 * len(diffs)) - 1]], 'replicates': len(diffs)}

def mcnemar_exact(b, c):
    """Two-sided exact McNemar p-value for discordant counts b and c."""
    n = b + c
    if n == 0: return 1.0
    k = min(b, c)
    p = sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, 2 * p)

def summarize(recs):
    res = {'n': len(recs), 'conditions': {c: cond_metrics(recs, c) for c in ('C0', 'C1', 'C2', 'C3')}}
    for c in res['conditions'].values():
        c['accepted_precision_wilson95'] = wilson(c['accepted_correct'], c['accepted'])
    res['paired'] = {f'{a}-{b}': {k: paired_bootstrap(recs, a, b, k)
                                  for k in ('whole_test_recall', 'incorrect_accepted_rate_of_all',
                                            'accepted_precision', 'coverage')}
                     for a, b in (('C3', 'C1'), ('C3', 'C2'), ('C1', 'C0'))}
    b = sum(r['lgc_correct'] and not r['top1_correct'] for r in recs)
    c = sum(r['top1_correct'] and not r['lgc_correct'] for r in recs)
    res['lgc_vs_top1_exact'] = {'top1_exact': sum(r['top1_correct'] for r in recs) / len(recs),
                                'lgc_exact': sum(r['lgc_correct'] for r in recs) / len(recs),
                                'lgc_only_correct': b, 'top1_only_correct': c, 'mcnemar_p': mcnemar_exact(b, c)}
    res['by_reference_in_lexicon'] = {str(v): {c: cond_metrics([r for r in recs if r['ref_in_lexicon'] == v], c)
                                               for c in ('C0', 'C1', 'C3')} for v in (True, False)
                                      if any(r['ref_in_lexicon'] == v for r in recs)}
    return res

In [ ]:
# ---- Data: the same verified archive and prepared Appendix K manifests ----
DATA = '/content/rxhandbd'
RX_SHA = 'c6371c0a3a89301c0b6b2eac1d2d49f6f3d8e559a7a71b338655dde0a6a63b51'
archive = f'{WORK_K}/RxHandBD-ML.zip'
assert sha256_file(archive) == RX_SHA, 'archive hash differs from Appendix H'
if not os.path.exists(DATA):
    with zipfile.ZipFile(archive) as z: z.extractall(DATA)
idx = index_images(DATA)
train_all = resolve(read_rows(find_split_csv(DATA, 'train')), idx)
test_all = resolve(read_rows(find_split_csv(DATA, 'test')), idx)
by_id = {r['record_id']: r for r in train_all}
paths = {k: f'{WORK_K}/manifests/k_{k}.csv' for k in ('train', 'val', 'holdout')}
EXPECTED_MAN = {'train': '32542934fd75f0f17a474c9074f595a2c78bd01648f056b255df7812efcb62a1',
                'val': 'da17c7247a4df45fc9b036a725c08383559ad732482876a5058efacb82187eb1',
                'holdout': 'fff7c370a8f8b5cc401535ce0fb2b5fd4cf35e76ee34e75358f4fc0334fb8dc8'}
for k, p in paths.items():
    assert sha256_file(p) == EXPECTED_MAN[k], f'{k} manifest differs from Appendix K'
splits = {k: [by_id[Path(n).name] for n, _ in read_rows(p)] for k, p in paths.items()}
splits = {k: [r for r in v if is_scorable(r['reference_text'])] for k, v in splits.items()}
test_primary = [r for r in test_all if is_scorable(r['reference_text'])]
print({k: len(v) for k, v in splits.items()}, 'original test', len(test_primary))   # 3,125/669/669 and 1,114
LEX_TRAIN = Lexicon([r['reference_text'] for r in splits['train']])
LEX_DEV = Lexicon([r['reference_text'] for r in splits['train'] + splits['val']])
print('lexicon entries: train', len(LEX_TRAIN.keys), '| train+val', len(LEX_DEV.keys))

In [ ]:
# ---- Recognizer: model loading, augmentation, training data, decoding, forced scoring ----
import random, time, math, shutil
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, get_linear_schedule_with_warmup
from huggingface_hub import HfApi
import torchvision.transforms as T
from PIL import ImageFilter

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

REV_P = f'{WORK}/model_revisions.json'
REVS = json.load(open(REV_P)) if os.path.exists(REV_P) else {}
for _, mid, _ in TRIALS:
    if mid not in REVS:
        REVS[mid] = 'eaacaf452b06415df8f10bb6fad3a4c11e609406' if mid.endswith('base-handwritten') else HfApi().model_info(mid).sha
json.dump(REVS, open(REV_P, 'w'), indent=1); print(REVS)

PROC = {}
def processor_for(mid):
    if mid not in PROC:
        PROC[mid] = TrOCRProcessor.from_pretrained(mid, revision=REVS[mid], use_fast=False)
    return PROC[mid]

def load_model(mid, path=None):
    m = VisionEncoderDecoderModel.from_pretrained(path or mid, revision=None if path else REVS[mid])
    tok = processor_for(mid).tokenizer
    if m.config.decoder_start_token_id is None:
        m.config.decoder_start_token_id = m.generation_config.decoder_start_token_id or tok.eos_token_id
    if m.config.pad_token_id is None:
        m.config.pad_token_id = tok.pad_token_id
    return m.cuda()

AFFINE = T.RandomAffine(degrees=3, translate=(0.03, 0.03), scale=(0.90, 1.05), shear=5, fill=255)
JITTER = T.ColorJitter(brightness=0.3, contrast=0.3)
def augment(im):
    if random.random() > 0.8: return im
    im = JITTER(AFFINE(im))
    if random.random() < 0.2: im = im.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.0)))
    if random.random() < 0.2: im = im.filter(random.choice([ImageFilter.MinFilter(3), ImageFilter.MaxFilter(3)]))
    return im

def prep(path):
    return crop_and_pad(Image.open(path).convert('RGB'))

class Words(Dataset):
    def __init__(self, recs, mid, train): self.recs, self.p, self.train = recs, processor_for(mid), train
    def __len__(self): return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]; im = prep(r['image_path'])
        if self.train: im = augment(im)
        px = self.p(images=im, return_tensors='pt').pixel_values[0]
        lab = self.p.tokenizer(r['reference_text'], max_length=MAX_LABEL_TOKENS, truncation=True).input_ids
        return px, lab

def collate(batch):
    px = torch.stack([b[0] for b in batch]); L = max(len(b[1]) for b in batch)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, (_, l) in enumerate(batch): lab[i, :len(l)] = torch.tensor(l)
    return px, lab

def pixel_batches(recs, mid, bs):
    p = processor_for(mid)
    for i in range(0, len(recs), bs):
        chunk = recs[i:i + bs]
        yield chunk, p(images=[prep(r['image_path']) for r in chunk], return_tensors='pt').pixel_values.cuda()

@torch.no_grad()
def predict(model, mid, recs, beams=1):
    model.eval(); tok = processor_for(mid).tokenizer; out = []
    for chunk, px in pixel_batches(recs, mid, EVAL_BATCH):
        with torch.autocast('cuda', dtype=DTYPE):
            ids = model.generate(px, max_new_tokens=MAX_NEW_TOKENS, num_beams=beams, do_sample=False, early_stopping=beams > 1)
        out += tok.batch_decode(ids, skip_special_tokens=True)
    return out

def evaluate(model, mid, recs, beams=1):
    hyps = predict(model, mid, recs, beams)
    return score([r['reference_text'] for r in recs], hyps), hyps

def better(a, b):
    if b is None: return True
    return (a['exact'], -a['wer']) > (b['exact'], -b['wer'])

@torch.no_grad()
def candidates(model, mid, recs, top1, lex):
    """n-best list + nearest lexicon entries, each scored by teacher-forced log P(text | image)."""
    model.eval(); tok = processor_for(mid).tokenizer; out = []; k = 0
    for chunk, px in pixel_batches(recs, mid, 8):
        with torch.autocast('cuda', dtype=DTYPE):
            ids = model.generate(px, max_new_tokens=MAX_NEW_TOKENS, num_beams=NBEST, num_return_sequences=NBEST,
                                 do_sample=False, early_stopping=True)
            enc = model.encoder(pixel_values=px).last_hidden_state
        nb = tok.batch_decode(ids, skip_special_tokens=True)
        for j, r in enumerate(chunk):
            t1 = top1[k]; k += 1
            texts, seen = [], set()
            for t in [t1] + nb[j * NBEST:(j + 1) * NBEST] + [lex.spelling(e) for e in lex.nearest(t1, KLEX)]:
                if t.strip() and t not in seen:
                    seen.add(t); texts.append(t)
            lab = tok(texts, padding=True, max_length=MAX_LABEL_TOKENS, truncation=True, return_tensors='pt').input_ids
            lab[lab == tok.pad_token_id] = -100
            hs = enc[j:j + 1].expand(len(texts), -1, -1)
            with torch.autocast('cuda', dtype=DTYPE):
                logits = model(encoder_outputs=(hs,), labels=lab.cuda()).logits.float()
            lp = torch.log_softmax(logits, -1)
            mask = lab.cuda() != -100
            tok_lp = lp.gather(-1, lab.cuda().clamp(min=0).unsqueeze(-1)).squeeze(-1) * mask
            cands = [{'text': t, 'logp': float(s), 'ntok': int(n)}
                     for t, s, n in zip(texts, tok_lp.sum(-1).tolist(), mask.sum(-1).tolist())]
            out.append({'id': r['record_id'], 'ref': r['reference_text'], 'top1': t1, 'cands': cands})
    return out

In [ ]:
# ---- Three recognizer trials; resumable after a disconnect (state saved each epoch) ----
def run_trial(name, mid, lr):
    tdir = f'{WORK}/trials/{name}'; os.makedirs(tdir, exist_ok=True)
    sp = f'{tdir}/status.json'
    st = json.load(open(sp)) if os.path.exists(sp) else {'epoch': 0, 'history': [], 'best': None, 'done': False, 'stale': 0}
    if st['done']:
        print(name, 'complete; best', st['best']); return st
    set_seed(SEED + st['epoch'])
    model = load_model(mid, f'{tdir}/latest' if st['epoch'] else None)
    dl = DataLoader(Words(splits['train'], mid, True), batch_size=BATCH, shuffle=True, collate_fn=collate,
                    num_workers=8, generator=torch.Generator().manual_seed(SEED + st['epoch']),
                    worker_init_fn=lambda w: random.seed(SEED * 1000 + st['epoch'] * 100 + w))
    spe = math.ceil(len(dl) / ACCUM); total = spe * MAX_EPOCHS
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WD)
    sch = get_linear_schedule_with_warmup(opt, int(WARMUP * total), total)
    if st['epoch']:
        ck = torch.load(f'{tdir}/latest/train_state.pt', map_location='cuda')
        opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch'])
    for epoch in range(st['epoch'] + 1, MAX_EPOCHS + 1):
        model.train(); t0 = time.time(); run = 0.0; opt.zero_grad(set_to_none=True)
        for step, (px, lab) in enumerate(dl, 1):
            with torch.autocast('cuda', dtype=DTYPE):
                loss = model(pixel_values=px.cuda(), labels=lab.cuda()).loss / ACCUM
            loss.backward(); run += loss.item()
            if step % ACCUM == 0 or step == len(dl):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        val, _ = evaluate(model, mid, splits['val'], 1)
        imp = better(val, st['best'])
        rec = {'epoch': epoch, 'train_loss': run / spe, 'val_exact': val['exact'], 'val_cer': val['cer'],
               'val_wer': val['wer'], 'improved': imp, 'minutes': (time.time() - t0) / 60}
        st['history'].append(rec); print(name, rec)
        if imp:
            st['best'] = {k: val[k] for k in ('exact', 'exact_count', 'cer', 'wer')} | {'epoch': epoch}
            model.save_pretrained(f'{tdir}/best'); st['stale'] = 0
        else:
            st['stale'] += 1
        model.save_pretrained(f'{tdir}/latest')
        torch.save({'opt': opt.state_dict(), 'sch': sch.state_dict()}, f'{tdir}/latest/train_state.pt')
        st['epoch'] = epoch
        if st['stale'] >= PATIENCE or epoch == MAX_EPOCHS:
            st['done'] = True; st['stopped_early'] = st['stale'] >= PATIENCE
        json.dump(st, open(sp, 'w'), indent=1)
        if st['done']: break
    shutil.rmtree(f'{tdir}/latest', ignore_errors=True)
    del model, opt; torch.cuda.empty_cache()
    return st

results = {name: run_trial(name, mid, lr) for name, mid, lr in TRIALS}

In [ ]:
# ---- Freeze stage 1 (validation only): recognizer trial, then greedy vs. beam 4 ----
FROZEN = f'{WORK}/frozen_selection.json'
if os.path.exists(FROZEN):
    fz = json.load(open(FROZEN)); print('Using the selection frozen earlier.')
else:
    best_name, best = None, None
    for name, st in results.items():
        if better(st['best'], best): best_name, best = name, st['best']
    mid = dict((n, m) for n, m, _ in TRIALS)[best_name]
    sel_dir = f'{WORK}/trials/{best_name}/best'
    model = load_model(mid, sel_dir); dec, hyp = {}, {}
    for b in (1, 4):
        dec[b], hyp[b] = evaluate(model, mid, splits['val'], b)
        print('validation beams', b, {k: dec[b][k] for k in ('exact', 'cer', 'wer')})
    BEAMS = 4 if better(dec[4], dec[1]) and dec[4] != dec[1] else 1
    # Stage 2 on validation: LGC weights, then the C3 margin threshold (training-label lexicon only).
    val_c = candidates(model, mid, splits['val'], hyp[BEAMS], LEX_TRAIN)
    with open(f'{WORK}/val_candidates.jsonl', 'w') as f:
        for r in val_c: f.write(json.dumps(r, ensure_ascii=False) + '\n')
    lgc_p, grid = select_lgc(val_c, LEX_TRAIN)
    theta, c1_prec, theta_rows = select_theta(val_c, LEX_TRAIN, lgc_p)
    val_summary = summarize(build_records(val_c, LEX_TRAIN, lgc_p, theta))
    fz = {'selected_trial': best_name, 'model_id': mid, 'revision': REVS[mid], 'epoch': best['epoch'],
          'decoding': {'num_beams': BEAMS, 'max_new_tokens': MAX_NEW_TOKENS},
          'lgc': {k: lgc_p[k] for k in ('alpha', 'beta', 'mode')} | {'nbest': NBEST, 'k_lexicon': KLEX},
          'theta': theta, 'c1_validation_precision': c1_prec,
          'validation': {'trials': {n: s['best'] for n, s in results.items()},
                         'decoding': {str(b): {x: dec[b][x] for x in ('exact', 'exact_count', 'cer', 'wer')} for b in dec},
                         'lgc_grid': grid, 'theta_rows': theta_rows, 'summary': val_summary},
          'checkpoint_sha256': {p.name: sha256_file(p) for p in sorted(Path(sel_dir).glob('*')) if p.is_file()},
          'manifest_sha256': EXPECTED_MAN, 'lexicon_entries': {'train': len(LEX_TRAIN.keys), 'dev': len(LEX_DEV.keys)},
          'protocol': {'trials': TRIALS, 'batch': BATCH, 'accum': ACCUM, 'max_epochs': MAX_EPOCHS, 'warmup': WARMUP,
                       'weight_decay': WD, 'patience': PATIENCE, 'seed': SEED, 'dtype': str(DTYPE)},
          'environment': {'torch': torch.__version__, 'transformers': transformers.__version__,
                          'gpu': torch.cuda.get_device_name(0), 'frozen_at': time.strftime('%Y-%m-%d %H:%M:%S %Z')}}
    json.dump(fz, open(FROZEN, 'w'), indent=1)
    del model; torch.cuda.empty_cache()
print('FROZEN:', fz['selected_trial'], fz['decoding'], fz['lgc'], 'theta', fz['theta'], '| sha256', sha256_file(FROZEN))
print('validation (train lexicon):', {c: round(v['whole_test_recall'], 4) for c, v in fz['validation']['summary']['conditions'].items()})

In [ ]:
# ---- Test populations, processed only now: holdout (primary, second use) and original test (secondary) ----
# Patch paired_bootstrap to handle None values gracefully if a metric is None due to zero accepted samples
import sys
import json

def patched_paired_bootstrap(recs, a, b, key, B=2000, seed=42):
    rng = random.Random(seed); n = len(recs); diffs = []
    for _ in range(B):
        s = [recs[rng.randrange(n)] for _ in range(n)]
        ma, mb = cond_metrics(s, a)[key], cond_metrics(s, b)[key]
        if ma is not None and mb is not None:
            diffs.append(ma - mb)
    diffs.sort()
    val_a = cond_metrics(recs, a)[key]
    val_b = cond_metrics(recs, b)[key]
    diff_val = (val_a if val_a is not None else 0.0) - (val_b if val_b is not None else 0.0)
    return {'diff': diff_val,
            'ci95': [diffs[int(0.025 * len(diffs))], diffs[int(0.975 * len(diffs)) - 1]] if diffs else [0.0, 0.0], 'replicates': len(diffs)}

# Apply the patch to the global namespace
globals()['paired_bootstrap'] = patched_paired_bootstrap

fz = json.load(open(FROZEN))
mid, BEAMS = fz['model_id'], fz['decoding']['num_beams']
model = load_model(mid, f"{WORK}/trials/{fz['selected_trial']}/best")
LGC_P = fz['lgc']
out = {}
for pop, recs in (('holdout', splits['holdout']), ('original_test', test_primary)):
    rec_score, top1 = evaluate(model, mid, recs, BEAMS)
    cands = candidates(model, mid, recs, top1, LEX_DEV)
    with open(f'{WORK}/{pop}_candidates.jsonl', 'w') as f:
        for r in cands: f.write(json.dumps(r, ensure_ascii=False) + '\n')
    R = build_records(cands, LEX_DEV, LGC_P, fz['theta'])
    with open(f'{WORK}/{pop}_records.jsonl', 'w') as f:
        for r in R: f.write(json.dumps(r, ensure_ascii=False) + '\n')
    lgc_score = score([r['ref'] for r in R], [r['lgc'] for r in R])
    def row(s): return {k: s[k] for k in ('n', 'exact_count', 'exact', 'cer', 'wer', 'char_counts', 'word_counts')} | \
                       {'exact_wilson95': wilson(s['exact_count'], s['n'])}
    out[pop] = {'recognizer_top1': row(rec_score), 'with_lgc': row(lgc_score), 'conditions': summarize(R)}
    print(pop, 'top1 exact', f"{rec_score['exact']:.2%}", 'CER', f"{rec_score['cer']:.2%}", 'WER', f"{rec_score['wer']:.2%}",
          '| with LGC exact', f"{lgc_score['exact']:.2%}", 'CER', f"{lgc_score['cer']:.2%}", 'WER', f"{lgc_score['wer']:.2%}")
    for c, v in out[pop]['conditions']['conditions'].items():
        print('  ', c, 'accepted', v['accepted'], 'incorrect', v['incorrect_accepted'], 'recall', f"{v['whole_test_recall']:.2%}",
              'review', f"{v['review_rate']:.2%}")

# Paired comparison with the Appendix K fine-tuned model on the same holdout images
prev = {json.loads(l)['record_id']: json.loads(l) for l in open(f'{WORK_K}/holdout_predictions.jsonl')}
recs_h = [json.loads(l) for l in open(f'{WORK}/holdout_records.jsonl')]
def ok(a, b): return normalize(a) == normalize(b)
cmp = {}
for key in ('top1', 'lgc'):
    b = sum(ok(r[key], r['ref']) and not ok(prev[r['id']]['finetuned'], r['ref']) for r in recs_h)
    c = sum(ok(prev[r['id']]['finetuned'], r['ref']) and not ok(r[key], r['ref']) for r in recs_h)
    cmp[key] = {'new_only_correct': b, 'appendix_k_only_correct': c, 'mcnemar_p': mcnemar_exact(b, c)}
out['vs_appendix_k_holdout'] = cmp
out['target'] = TARGET
out['target_met'] = {k: out['holdout'][k]['exact'] >= TARGET for k in ('recognizer_top1', 'with_lgc')}
print('vs Appendix K:', cmp, '| 80% target met:', out['target_met'])
json.dump(out, open(f'{WORK}/test_results.json', 'w'), indent=1)

In [ ]:
# ---- Package for the praxis ----
from google.colab import files
import glob
bundle = {'frozen_selection': fz, 'results': out, 'trial_histories': {n: s['history'] for n, s in results.items()},
          'frozen_selection_sha256': sha256_file(FROZEN)}
json.dump(bundle, open(f'{WORK}/appendix_m_results.json', 'w'), indent=1)
zp = f'{WORK}/appendix_m_results.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in ['appendix_m_results.json', 'frozen_selection.json', 'test_results.json', 'model_revisions.json',
              'val_candidates.jsonl', 'holdout_candidates.jsonl', 'original_test_candidates.jsonl',
              'holdout_records.jsonl', 'original_test_records.jsonl']:
        z.write(f'{WORK}/{p}', p)
print('saved', zp, os.path.getsize(zp) // 1024, 'KB')
files.download(zp)

In [ ]:
# ---- Correction run (added after the first test evaluation) ----
# Defect found: candidates() scored each text with the tokenizer's leading <s> token, which this checkpoint never
# generates (generation emits </s> -> text -> </s>). Every candidate was therefore scored from an impossible prefix
# (log P about -40), which made the LGC rescoring meaningless. The recognizer and its top-1 outputs are unaffected.
# Correction: drop the leading <s> before teacher-forced scoring. Stage 1 (recognizer, decoding) stays frozen as is;
# stage 2 (LGC weights, theta) is re-selected on VALIDATION ONLY with the same pre-specified grid and rule, frozen
# as frozen_selection_v2.json, and only then are the holdout and original test re-scored. Both runs are reported.
import shutil
def strip_bos(lab, tok):
    return lab[:, 1:] if bool((lab[:, 0] == tok.bos_token_id).all()) else lab

@torch.no_grad()
def candidates_v2(model, mid, recs, top1, lex):
    model.eval(); tok = processor_for(mid).tokenizer; out = []; k = 0
    for chunk, px in pixel_batches(recs, mid, 8):
        with torch.autocast('cuda', dtype=DTYPE):
            ids = model.generate(px, max_new_tokens=MAX_NEW_TOKENS, num_beams=NBEST, num_return_sequences=NBEST,
                                 do_sample=False, early_stopping=True)
            enc = model.encoder(pixel_values=px).last_hidden_state
        nb = tok.batch_decode(ids, skip_special_tokens=True)
        for j, r in enumerate(chunk):
            t1 = top1[k]; k += 1
            texts, seen = [], set()
            for t in [t1] + nb[j * NBEST:(j + 1) * NBEST] + [lex.spelling(e) for e in lex.nearest(t1, KLEX)]:
                if t.strip() and t not in seen:
                    seen.add(t); texts.append(t)
            lab = tok(texts, padding=True, max_length=MAX_LABEL_TOKENS, truncation=True, return_tensors='pt').input_ids
            lab = strip_bos(lab, tok).clone(); lab[lab == tok.pad_token_id] = -100
            hs = enc[j:j + 1].expand(len(texts), -1, -1)
            with torch.autocast('cuda', dtype=DTYPE):
                logits = model(encoder_outputs=(hs,), labels=lab.cuda()).logits.float()
            lp = torch.log_softmax(logits, -1); mask = lab.cuda() != -100
            tok_lp = lp.gather(-1, lab.cuda().clamp(min=0).unsqueeze(-1)).squeeze(-1) * mask
            out.append({'id': r['record_id'], 'ref': r['reference_text'], 'top1': t1,
                        'cands': [{'text': t, 'logp': float(s), 'ntok': int(n)}
                                  for t, s, n in zip(texts, tok_lp.sum(-1).tolist(), mask.sum(-1).tolist())]})
    return out

fz1 = json.load(open(FROZEN)); mid, BEAMS = fz1['model_id'], fz1['decoding']['num_beams']
model = load_model(mid, f"{WORK}/trials/{fz1['selected_trial']}/best")
FROZEN2 = f'{WORK}/frozen_selection_v2.json'
if os.path.exists(FROZEN2):
    fz2 = json.load(open(FROZEN2)); print('Using the v2 selection frozen earlier.')
else:
    _, vtop = evaluate(model, mid, splits['val'], BEAMS)
    val_c = candidates_v2(model, mid, splits['val'], vtop, LEX_TRAIN)
    with open(f'{WORK}/val_candidates_v2.jsonl', 'w') as f:
        for r in val_c: f.write(json.dumps(r, ensure_ascii=False) + '\n')
    lgc_p, grid = select_lgc(val_c, LEX_TRAIN)
    theta, c1_prec, theta_rows = select_theta(val_c, LEX_TRAIN, lgc_p)
    fz2 = dict(fz1); fz2['lgc'] = {k: lgc_p[k] for k in ('alpha', 'beta', 'mode')} | {'nbest': NBEST, 'k_lexicon': KLEX}
    fz2['theta'] = theta; fz2['c1_validation_precision'] = c1_prec
    fz2['validation'] = dict(fz1['validation']) | {'lgc_grid': grid, 'theta_rows': theta_rows,
                                                    'summary': summarize(build_records(val_c, LEX_TRAIN, lgc_p, theta))}
    fz2['correction'] = {'defect': 'leading <s> included in teacher-forced candidate scoring',
                         'v1_sha256': sha256_file(FROZEN), 'frozen_at': time.strftime('%Y-%m-%d %H:%M:%S %Z')}
    json.dump(fz2, open(FROZEN2, 'w'), indent=1)
print('FROZEN v2:', fz2['lgc'], 'theta', fz2['theta'], '| sha256', sha256_file(FROZEN2))
print('validation:', fz2['validation']['summary']['lgc_vs_top1_exact'])

out2 = {}
for pop, recs in (('holdout', splits['holdout']), ('original_test', test_primary)):
    top1 = [json.loads(l)['top1'] for l in open(f'{WORK}/{pop}_records.jsonl')]   # frozen recognizer outputs
    cands = candidates_v2(model, mid, recs, top1, LEX_DEV)
    with open(f'{WORK}/{pop}_candidates_v2.jsonl', 'w') as f:
        for r in cands: f.write(json.dumps(r, ensure_ascii=False) + '\n')
    R = build_records(cands, LEX_DEV, fz2['lgc'], fz2['theta'])
    with open(f'{WORK}/{pop}_records_v2.jsonl', 'w') as f:
        for r in R: f.write(json.dumps(r, ensure_ascii=False) + '\n')
    s = score([r['ref'] for r in R], [r['lgc'] for r in R])
    out2[pop] = {'with_lgc': {k: s[k] for k in ('n', 'exact_count', 'exact', 'cer', 'wer', 'char_counts', 'word_counts')}
                 | {'exact_wilson95': wilson(s['exact_count'], s['n'])}, 'conditions': summarize(R)}
    print(pop, 'LGC exact', f"{s['exact']:.2%}", 'CER', f"{s['cer']:.2%}", 'WER', f"{s['wer']:.2%}")
    for cnd, v in out2[pop]['conditions']['conditions'].items():
        print('  ', cnd, 'accepted', v['accepted'], 'incorrect', v['incorrect_accepted'],
              'recall', f"{v['whole_test_recall']:.2%}", 'review', f"{v['review_rate']:.2%}")
json.dump(out2, open(f'{WORK}/test_results_v2.json', 'w'), indent=1)
zp = f'{WORK}/appendix_m_results_v2.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in ['frozen_selection_v2.json', 'test_results_v2.json', 'val_candidates_v2.jsonl', 'holdout_candidates_v2.jsonl',
              'original_test_candidates_v2.jsonl', 'holdout_records_v2.jsonl', 'original_test_records_v2.jsonl']:
        z.write(f'{WORK}/{p}', p)
from google.colab import files
files.download(zp)